# Data preparation: Household Pulse Survey, weeks 22–34 (2021)

Replication of Shafer et al. (2022), *Association of the Implementation of Child Tax Credit Advance Payments With Food Insufficiency in US Households*.

This notebook builds the analysis files from the raw Census downloads using the functions in `modules/`:

1. Download each week's CSV zip (optional; or place the zips in `data/raw/` yourself)
2. Unzip them
3. Check which variables each week contains
4. Stack the weeks and build derived variables
5. Save the processed files and run the first replication checks

**Raw data:** [Census 2021 Household Pulse PUF page](https://www.census.gov/programs-surveys/household-pulse-survey/data/datasets.2021.html), weeks 22–34, CSV format. Leave each zip named as Census names it (e.g. `HPS_Week22_PUF_CSV.zip`).

## Setup

In [1]:
import sys
from pathlib import Path

# Make the repo root importable whether the notebook runs from the repo root or notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from modules import get_data, prepare_pulse

print("Repo root:", ROOT)
print("Raw data folder:", get_data.RAW_DIR)

Repo root: c:\Users\Rosey\Desktop\CTC-Food-Insufficiency
Raw data folder: C:\Users\Rosey\Desktop\CTC-Food-Insufficiency\data\raw


## 1. Download the raw zips (optional)

Set `DOWNLOAD = True` to fetch the 13 weekly zips from Census (about 1 GB in total). Files already in `data/raw/` are skipped. If you downloaded the zips manually, leave this `False`.

In [2]:
DOWNLOAD = False

if DOWNLOAD:
    failed = get_data.download_pulse()

## 2. Unzip each week into `data/raw/HPS_Week<NN>_PUF_CSV/`

In [3]:
status = get_data.unzip_pulse()
missing_zips = [w for w, s in status.items() if s == "MISSING ZIP"]
assert not missing_zips, f"Missing zips for weeks {missing_zips}: download them before continuing."

week 22: extracted
week 23: extracted
week 24: extracted
week 25: extracted
week 26: extracted
week 27: extracted
week 28: extracted
week 29: extracted
week 30: extracted
week 31: extracted
week 32: extracted
week 33: extracted
week 34: extracted


## 3. Variable check across weeks

Which source column each harmonized variable comes from in each week. Names change between phases (e.g. `EGENDER` → `EGENID_BIRTH` and `SPNDSRC5` → `SPND_SRC5` in week 34). Any `— missing —` cell needs a decision before modeling.

In [2]:
report = prepare_pulse.variable_report()
missing = report.columns[(report == "— missing —").any()]
print("Variables missing in at least one week:", list(missing) or "none")
report[missing] if len(missing) else report

Variables missing in at least one week: none


,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,UI_SPND,STIM_SPND,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
week,,,,,,,,,,,,,,,,,,,,,
22,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
23,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
24,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
25,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
26,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
27,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
28,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
29,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8
30,SCRAM,WEEK,EST_ST,HWEIGHT,CURFOODSUF,THHLD_NUMKID,THHLD_NUMADLT,TBIRTH_YEAR,RRACE,RHISPANIC,...,SPNDSRC5,SPNDSRC6,HLTHINS1,HLTHINS2,HLTHINS3,HLTHINS4,HLTHINS5,HLTHINS6,HLTHINS7,HLTHINS8


## 4. Stack weeks and build derived variables

In [3]:
data, missing_by_week = prepare_pulse.build_analysis_file()
print(data.shape)
data[["WEEK", "food_insuf", "kids", "post", "kids_post", "age", "female", "w"]].head()

week 22:  68,348 rows | sex from EGENDER | missing vars: none
week 23:  80,567 rows | sex from EGENDER | missing vars: none
week 24:  77,122 rows | sex from EGENDER | missing vars: none
week 25:  77,788 rows | sex from EGENDER | missing vars: none
week 26:  78,306 rows | sex from EGENDER | missing vars: none
week 27:  77,104 rows | sex from EGENDER | missing vars: none
week 28:  68,913 rows | sex from EGENDER | missing vars: none
week 29:  78,467 rows | sex from EGENDER | missing vars: none
week 30:  72,897 rows | sex from EGENDER | missing vars: none
week 31:  70,854 rows | sex from EGENDER | missing vars: none
week 32:  68,067 rows | sex from EGENDER | missing vars: none
week 33:  66,262 rows | sex from EGENDER | missing vars: none
week 34:  64,562 rows | sex from EGENID_BIRTH | missing vars: none
(949257, 36)


,WEEK,food_insuf,kids,post,kids_post,age,female,w
0,22,NaN,1,0,0,43,1,90.060570
1,22,0.0,0,0,0,74,1,69.228380
2,22,0.0,1,0,0,32,0,159.833558
3,22,0.0,0,0,0,50,1,273.493925
4,22,0.0,0,0,0,36,1,91.351105


### Missingness by week (key variables)

Share of rows missing each variable in each week. Large jumps at week 34 would point to the Phase 3.2 questionnaire change.

In [4]:
key = ["food_insuf", "INCOME", "female", "EEDUC", "ANYWORK", "SNAP_YN", "UI_SPND", "STIM_SPND"]
key = [c for c in key if c in data]
(data.groupby("WEEK")[key].apply(lambda g: g.isna().mean()) * 100).round(1)

,food_insuf,INCOME,female,EEDUC,ANYWORK,SNAP_YN,UI_SPND,STIM_SPND
WEEK,,,,,,,,
22,11.0,24.4,0.0,0.0,1.2,12.0,9.8,9.8
23,9.7,22.5,0.0,0.0,0.7,10.6,8.6,8.6
24,10.2,23.0,0.0,0.0,0.7,11.1,9.1,9.1
25,10.2,22.8,0.0,0.0,0.7,11.1,9.0,9.0
26,10.9,24.0,0.0,0.0,0.7,11.8,9.7,9.7
27,10.7,23.9,0.0,0.0,0.6,11.8,9.6,9.6
28,12.1,27.0,0.0,0.0,0.7,13.1,10.8,10.8
29,11.4,26.3,0.0,0.0,0.7,12.5,10.1,10.1
30,11.5,26.4,0.0,0.0,0.7,12.6,10.2,10.2


## 5. Replicate weights (for Extension b)

Stacked separately as 32-bit floats; each week is checked for a 1:1 match on `SCRAM`.

In [5]:
repwgt = prepare_pulse.build_repwgt_file()
repwgt.shape

week 22:  68,348 replicate rows | unmatched: 0
week 23:  80,567 replicate rows | unmatched: 0
week 24:  77,122 replicate rows | unmatched: 0
week 25:  77,788 replicate rows | unmatched: 0
week 26:  78,306 replicate rows | unmatched: 0
week 27:  77,104 replicate rows | unmatched: 0
week 28:  68,913 replicate rows | unmatched: 0
week 29:  78,467 replicate rows | unmatched: 0
week 30:  72,897 replicate rows | unmatched: 0
week 31:  70,854 replicate rows | unmatched: 0
week 32:  68,067 replicate rows | unmatched: 0
week 33:  66,262 replicate rows | unmatched: 0
week 34:  64,562 replicate rows | unmatched: 0


(949257, 82)

## 6. Save processed files

In [6]:
prepare_pulse.save(data, "pulse_w22_34")
prepare_pulse.save(repwgt, "repwgt_w22_34")

saved C:\Users\Rosey\Desktop\CTC-Food-Insufficiency\data\processed\pulse_w22_34.parquet
saved C:\Users\Rosey\Desktop\CTC-Food-Insufficiency\data\processed\repwgt_w22_34.parquet


WindowsPath('C:/Users/Rosey/Desktop/CTC-Food-Insufficiency/data/processed/repwgt_w22_34.parquet')

## 7. First replication checks

Targets from Shafer et al. (2022): 667,836 respondents under 65 before exclusions; weighted food insufficiency among households with children of 14.3% in week 33 and 9.9% in week 34.

In [7]:
prepare_pulse.replication_checks(data)

,check,ours,paper
0,"Respondents under 65, all weeks (before exclus...","667,836","667,836"
1,"Food insufficiency, households with kids, week...",14.4%,14.3%
2,"Food insufficiency, households with kids, week...",10.0%,9.9%


Need to investigate the slight difference in results, this might be due to a sampling difference so let's investigate

In [8]:
# compare 
a = prepare_pulse.analytic_sample(data)
print(f"{len(a):,}  (paper: 585,170)")
k = a[a.kids == 1]
for wk in (33, 34):
    s = k[k.WEEK == wk]
    print(wk, round(100 * np.average(s.food_insuf, weights=s.HWEIGHT), 1))

585,170  (paper: 585,170)
33 14.3
34 9.9


In [9]:
base = data[(data.age < 65) & data.food_insuf.notna()]
print(f"after food_insuf: {len(base):,}  (paper total: 585,170)")
for c in prepare_pulse.CONTROLS:
    print(f"{c:15s} drops {base[c].isna().sum():>7,} on its own")

after food_insuf: 587,966  (paper total: 585,170)
female          drops       0 on its own
age             drops       0 on its own
RRACE           drops       0 on its own
RHISPANIC       drops       0 on its own
EEDUC           drops       0 on its own
MS              drops   2,796 on its own
THHLD_NUMADLT   drops       0 on its own
THHLD_NUMKID    drops       0 on its own
ANYWORK         drops     895 on its own
UI_SPND         drops       0 on its own
STIM_SPND       drops       0 on its own
SNAP_YN         drops   7,069 on its own
FREEFOOD        drops   3,573 on its own
EST_ST          drops       0 on its own
insured         drops  51,949 on its own


So, we need to lose exactly **2,796** rows after the food question (587,966 − 585,170), and marital status drops **exactly 2,796** on its own. That's unlikely to be a coincidence.

My best guess to what the paper did:
- **Required to be non-missing:** the outcome and marital status. Everything else is either never missing or overlaps with those rows.
- **Kept with a "missing" category:** the other controls with gaps (insurance, SNAP, food aid, employment), the same way income was handled. That's common in JAMA-style tables, which often list a "missing" row for each covariate.

In [10]:
# testing the effect of dropping MS, which is missing for 0.5% of the sample
only_ms = base.dropna(subset=["MS"])
print(f"food_insuf + MS only: {len(only_ms):,}  (paper: 585,170)")

k = only_ms[only_ms.kids == 1]
for wk in (33, 34):
    s = k[k.WEEK == wk]
    print(wk, round(100 * np.average(s.food_insuf, weights=s.HWEIGHT), 1), "(paper:", {33: 14.3, 34: 9.9}[wk], ")")

food_insuf + MS only: 585,170  (paper: 585,170)
33 14.3 (paper: 14.3 )
34 9.9 (paper: 9.9 )


The tests confirms the sample definition. Adjusted `prepare_pulse.py` in modules to make the changes necessary to get the correct sample! 

## 8. Save final sample

The paper says it used complete-case analysis except for income. In practice, the only rows dropped beyond the outcome were those missing marital status, so the other covariates with gaps must have kept missing as a category. That's an interesting replication finding, the kind of undocumented choice replications exist to surface.

In [11]:
#save the analytic sample for use in the notebook
a = prepare_pulse.analytic_sample(data)
assert len(a) == 585_170
prepare_pulse.save(a, "analytic_sample")

saved C:\Users\Rosey\Desktop\CTC-Food-Insufficiency\data\processed\analytic_sample.parquet


WindowsPath('C:/Users/Rosey/Desktop/CTC-Food-Insufficiency/data/processed/analytic_sample.parquet')